# HAR Stage 19A2 — independent human-genome sequence and sgRNA-flank check (no Google Drive)

**Objective:** check 152 provisional hg38 HAR catalog sequences against the canonical hg38 genome reference and locate the original 370 GEO CRISPRi 20-mer guides within a 250-bp flank on either side.

- Input: the `HAR_Stage19A_Colab_Source_Verified_Results.zip` from **Stage 19A**, which you already generated. No Google Drive is needed.
- Retrieves 152 reference slices from the [UCSC Genome API](https://api.genome.ucsc.edu/), using **hg38** chromosome coordinates and BED-like 0-based half-open intervals.
- Tests exact direct/reverse-complement matches for 20-nt guides and reports per-HAR and per-guide status, including genome reference identity and optional NGG PAM context.
- **Interpretation:** these results establish genome-sequence consistency and local guide placement only. They do **not** establish the original Noble authors' exact HAR boundaries, experimental target-gene assignment, TF occupancy or functional causality.
- Output: a single small `HAR_Stage19A2_Flank_Guide_Results.zip`, with status JSON and complete CSVs. If downloads fail, the report identifies failed coordinates rather than inventing successes.


In [ ]:
import csv, io, zipfile, json, hashlib, time, sys
from pathlib import Path
from collections import defaultdict, Counter
import requests

WORK=Path('/content') if Path('/content').exists() else Path.cwd()
SOURCE=WORK/'HAR_Stage19A_Colab_Source_Verified_Results.zip'
if not SOURCE.exists():
    try:
        from google.colab import files
        print('Please select your downloaded HAR_Stage19A_Colab_Source_Verified_Results.zip')
        uploaded=files.upload()
        choices=[n for n in uploaded if n.endswith('.zip')]
        if len(choices)!=1: raise RuntimeError('Upload exactly one Stage 19A results ZIP')
        SOURCE=WORK/choices[0]
        SOURCE.write_bytes(uploaded[choices[0]])
    except ImportError:
        raise FileNotFoundError('Please copy Stage19A results ZIP to this working directory and re-run')

assert SOURCE.exists() and zipfile.is_zipfile(SOURCE)
OUT=WORK/'HAR_Stage19A2_Outputs'
OUT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(SOURCE) as Z:
    def csv_read(name): return list(csv.DictReader(io.StringIO(Z.read(name).decode('utf-8-sig'))))
    targets=csv_read('S2_152_Provisional_Ortholog_Coordinates.csv')
    guide_rows=csv_read('GSE270828_feature_README.csv')
    manifest=csv_read('S9_152_Sequence_Checksum_Manifest.csv')
    def fasta(name):
        result={}; key=None; seq=[]
        for line in Z.read(name).decode().splitlines():
            if line.startswith('>'):
                if key is not None: result[key]=''.join(seq).upper()
                key=line[1:].split('|')[0]; seq=[]
            else: seq.append(line.strip())
        if key is not None: result[key]=''.join(seq).upper()
        return result
    human=fasta('S7_152_Human_CatalogSequences_PROVISIONAL.fa')

assert len(targets)==len(manifest)==len(human)==152 and len(guide_rows)==370
assert {r['screen_HAR'] for r in targets}==set(human)
for row in manifest:
    key=row['screen_HAR']
    assert hashlib.sha256(human[key].encode()).hexdigest()==row['human_seq_sha256'],f'Stage19A sequence hash mismatch: {key}'
print('Stage 19A extracted data verified: 152 targets, 152 sequences, 370 original guide references.')


In [ ]:
# Download hg38 chromosome sequence windows from UCSC. 250 bp on both sides of catalog interval.
# Keep a local JSONL checkpoint if the runtime remains active, so rerunning does not lose completed work.
FLANK=250
CACHE=OUT/'ucsc_hg38_cache.jsonl'
cache={}
if CACHE.exists():
    for line in CACHE.read_text().splitlines():
        try:
            record=json.loads(line)
            if record.get('dna'): cache[record['key']]=record
        except ValueError: pass

session=requests.Session()
session.headers.update({'User-Agent':'HAR-Stage19A2-Reproducible-Genome-Crosscheck/1.0 (public research)'})

def genome_slice(chrom,start,end):
    key=f'{chrom}:{start}-{end}'
    if key in cache: return cache[key]
    url='https://api.genome.ucsc.edu/getData/sequence'
    last=''
    for wait in [0,1,3,8,15]:
        if wait: time.sleep(wait)
        try:
            reply=session.get(url,params={'genome':'hg38','chrom':chrom,'start':start,'end':end},timeout=60)
            reply.raise_for_status()
            payload=reply.json()
            seq=str(payload.get('dna','')).upper()
            if len(seq)!=(end-start):
                raise ValueError(f'Genome sequence length mismatch: {len(seq)} != {end-start}')
            record={'key':key,'chrom':chrom,'start0':start,'end_exclusive':end,'dna':seq,'source_url':reply.url}
            with CACHE.open('a') as f: f.write(json.dumps(record)+'\n')
            cache[key]=record
            return record
        except Exception as exc: last=f'{type(exc).__name__}: {exc}'
    return {'key':key,'dna':None,'error':last}

fetched={}
failed=[]
for i,row in enumerate(targets,1):
    har=row['screen_HAR']; chrom=row['human_hg38_chrom']; start=int(row['human_start_0based']);end=int(row['human_end_exclusive'])
    left=max(0,start-FLANK);right=end+FLANK
    rec=genome_slice(chrom,left,right)
    fetched[har]=(rec,left,start,end,right)
    if rec.get('dna') is None: failed.append(har)
    if i%20==0 or i==len(targets): print(f'Reference slices processed: {i}/{len(targets)} | failed: {len(failed)}')
    if i%10==0: time.sleep(0.4) # modest request pacing
print('Fetched:',len(fetched)-len(failed),'Failed:',len(failed))


In [ ]:
COMP=str.maketrans('ACGT','TGCA')
def revcomp(x): return x.translate(COMP)[::-1]
def locate(text, spacer):
    if not text or len(spacer)!=20 or any(b not in 'ACGT' for b in spacer.upper()): return []
    answer=[]
    for strand,pattern in [('+',spacer.upper()),('-',revcomp(spacer.upper()))]:
        off=0
        while True:
            pos=text.find(pattern,off)
            if pos<0: break
            pam=None
            if strand=='+' and pos+23<=len(text): pam=text[pos+20:pos+23]
            if strand=='-' and pos>=3: pam=revcomp(text[pos-3:pos])
            answer.append({'offset0':pos,'strand':strand,'pam':pam,'canonical_NGG':bool(pam and len(pam)==3 and pam[1:]=='GG')})
            off=pos+1
    return answer

by_target=defaultdict(list)
for gr in guide_rows: by_target[gr['name']].append(gr)
rows=[]; per_guides=[]
for r in targets:
    k=r['screen_HAR']; ref,left,start,end,right=fetched[k]; seq=ref.get('dna'); core=None
    if seq is not None:
        core=seq[start-left:end-left]
        assert len(core)==end-start
    exact=(core==human[k]) if core is not None else None
    rc_exact=(core==revcomp(human[k])) if core is not None else None
    covered=Counter(); positions=[]
    for gr in by_target[k]:
        hits=locate(seq,gr['sequence'])
        locs=[]
        for h in hits:
            hit_start0=left+h['offset0'];hit_end0=hit_start0+20
            if hit_start0>=start and hit_end0<=end: relation='INSIDE_CORE'
            elif hit_end0<=start: relation='UPSTREAM'
            elif hit_start0>=end: relation='DOWNSTREAM'
            else: relation='CROSSES_CORE_BOUNDARY'
            locs.append({'genome_start0':hit_start0,'strand':h['strand'],'relative_to_core':relation,'canonical_NGG':h['canonical_NGG']})
        if locs: covered['matched']+=1
        if any(x['relative_to_core']=='INSIDE_CORE' for x in locs):covered['inside']+=1
        if any(x['relative_to_core']!='INSIDE_CORE' for x in locs):covered['flank']+=1
        positions.extend(locs)
        per_guides.append({
            'HAR':k,'guide_id':gr['id'],'original_guide_sequence':gr['sequence'],
            'source_slice_status':'FETCHED' if seq else 'FAILED',
            'n_matches_in_plusminus250_window':len(locs),
            'matching_coordinates_0based':'; '.join(f"{x['genome_start0']}:{x['strand']}:{x['relative_to_core']}" for x in locs),
            'canonical_NGG_any':any(x['canonical_NGG'] for x in locs),
            'interpretation':'Guide location does not certify author HAR boundaries'
        })
    rows.append({
        'HAR':k,'catalog_ID_provisional':r['catalog_ID_not_original_name'],
        'hg38_chrom':r['human_hg38_chrom'],'core_start0':start,'core_end_exclusive':end,
        'retrieval_start0':left,'retrieval_end_exclusive':right,
        'reference_status':'FETCHED' if seq else 'FAILED','error':ref.get('error',''),
        'core_exact_hg38_genome_match':exact,'core_reverse_complement_match':rc_exact,
        'n_original_guides':len(by_target[k]),
        'n_guides_found_within_plusminus250':covered['matched'] if seq else None,
        'n_guides_found_inside_core':covered['inside'] if seq else None,
        'n_guides_found_in_flanks_or_crossing':covered['flank'] if seq else None,
        'n_total_guide_alignment_sites':len(positions) if seq else None,
        'original_author_exact_HAR_intervals_confirmed':'NO',
        'biological_status':'PROVISIONAL_SEQUENCE_CONSISTENCY_ONLY'
    })

def save_csv(path,records):
    assert records
    with path.open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=list(records[0]));w.writeheader();w.writerows(records)
save_csv(OUT/'S1_152_Assembly_and_GuideFlank_Crosscheck.csv',rows)
save_csv(OUT/'S2_Original_GuideFlank_Alignment_Details.csv',per_guides)
summary={
    'stage':'19A2','genome_assembly':'hg38','flank_bp_each_side':FLANK,
    'original_screen_HARs':180,'provisional_sequence_pairs_examined':152,
    'genome_slices_succeeded':sum(r['reference_status']=='FETCHED' for r in rows),
    'genome_slices_failed':len(failed),'failed_HAR_ids':failed,
    'exact_hg38_core_match_count':sum(r['core_exact_hg38_genome_match'] is True for r in rows),
    'reverse_complement_hg38_core_match_count':sum(r['core_reverse_complement_match'] is True for r in rows),
    'HARs_with_at_least_one_guide_matched_window':sum((r['n_guides_found_within_plusminus250'] or 0)>0 for r in rows),
    'HARs_with_both_guides_matched_window':sum((r['n_guides_found_within_plusminus250'] or 0)==2 for r in rows),
    'missing_original_author_BED':True,
    'important':'Sequence/guide matches do not independently validate original HAR boundaries; no gene target or TF binding inferred'
}
(OUT/'STAGE19A2_STATUS.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))

RESULT=WORK/'HAR_Stage19A2_Flank_Guide_Results.zip'
with zipfile.ZipFile(RESULT,'w',zipfile.ZIP_DEFLATED,compresslevel=8) as z:
    for name in ['S1_152_Assembly_and_GuideFlank_Crosscheck.csv','S2_Original_GuideFlank_Alignment_Details.csv','STAGE19A2_STATUS.json']:
        z.write(OUT/name,arcname=name)
print('Created:',RESULT,'bytes:',RESULT.stat().st_size)
try:
    from google.colab import files
    files.download(str(RESULT))
except ImportError:
    print('Output ZIP saved at:',RESULT)
